## Lab 1

- Dado un video, mostrar los vectores de flujo en tiempo real utilizando great features to view OpenCV en Python.
- Luego, segmentar de manera automática el video utilizando técnicas de segmentación basadas en el flujo óptico.
- Tarea: presentación del algoritmo de optical flow de farneback


In [1]:
# pylint: skip-file
import cv2
import numpy as np

In [ ]:
# Sube el archivo cat_running.mp4 desde tu computadora hacia Google Colab
from google.colab import files

print("Por favor, sube el archivo cat_running.mp4:")
uploaded = files.upload()

# Obtener exactamente el nombre del archivo recién subido
video_path = list(uploaded.keys())[0]

In [2]:
# Parámetros para la detección de características (Shi-Tomasi)
feature_params = dict(maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)

# Parámetros para el flujo óptico de Lucas-Kanade
lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Error: No se puede abrir {video_path}")
else:
    # Propiedades del video original para el archivo de salida
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    # Crear el escritor de video en mp4
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out_path = "motion_out_" + video_path
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    ret, old_frame = cap.read()
    if ret:
        old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

        # Encontrar las mejores características en el primer frame
        p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

        # Colores aleatorios para dibujar
        c_mask = np.random.randint(0, 255, (100, 3))

        # Máscara para dibujar el recorrido
        mask = np.zeros_like(old_frame)

        print("Procesando video de cuadro por cuadro... por favor espera.")
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            img = frame.copy()

            # Calcular el flujo óptico solo si hay puntos para rastrear
            if p0 is not None and len(p0) > 0:
                p1, st, err = cv2.calcOpticalFlowPyrLK(
                    old_gray, frame_gray, p0, None, **lk_params
                )

                # Filtrar puntos válidos
                if p1 is not None and st is not None:
                    good_new = p1[st == 1]
                    good_old = p0[st == 1]

                    for i, (new, old) in enumerate(zip(good_new, good_old)):
                        a, b = new.ravel()
                        c, d = old.ravel()

                        mask = cv2.line(
                            mask,
                            (int(a), int(b)),
                            (int(c), int(d)),
                            c_mask[i].tolist(),
                            2,
                        )
                        img = cv2.circle(
                            img, (int(a), int(b)), 5, c_mask[i].tolist(), -1
                        )

                    img = cv2.add(img, mask)
                    p0 = good_new.reshape(-1, 1, 2)

                    # Si perdemos todos los puntos, podemos intentar redetectar
                    if len(p0) == 0:
                        p0 = cv2.goodFeaturesToTrack(
                            frame_gray, mask=None, **feature_params
                        )

            # Escribir el frame modificado en el nuevo archivo de video
            out.write(img)

            old_gray = frame_gray.copy()

        cap.release()
        out.release()
        print(f"\nProcesamiento terminado. El video se ha guardado como '{out_path}'.")

        # Descargar el video de regreso a tu PC automáticamente
        print(f"Descargando {out_path}...")
        files.download(out_path)
    else:
        print("No se pudo leer el primer frame del video.")

NameError: name 'video_path' is not defined